# Import Library

In [1]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

# Load dan Preprocess Earthquake

In [ ]:
# Load dan Preprocess Earthquake (USGS)

# ── Load file ─────────────────────────────────────────────────────────────────
eq_raw = pd.read_csv("../data/raw/weather/usgs_bali_earthquakes_2009_present.csv")

# ── Parse & bersihkan ────────────────────────────────────────────────────────
eq = eq_raw.copy()
eq["date"] = pd.to_datetime(eq["date"])

# Drop kolom redundant / konstan
# - "source"  : satu nilai konstan (USGS Earthquake API)
# - "year"    : derivable dari date, tidak perlu disimpan
eq = eq.drop(columns=["source", "year"])

# Hitung ulang distance_to_bali_km — kolom original memiliki selisih rata-rata
# ~9 km dari kalkulasi Haversine (kemungkinan reference point berbeda di sumber).
# Reference: koordinat pusat Bali (BMKG Denpasar area)
BALI_LAT, BALI_LON = -8.3405, 115.0920

def haversine_km(lat, lon, ref_lat=BALI_LAT, ref_lon=BALI_LON):
    R = 6371.0
    dlat = np.radians(ref_lat - lat)
    dlon = np.radians(ref_lon - lon)
    a = (np.sin(dlat/2)**2
         + np.cos(np.radians(lat)) * np.cos(np.radians(ref_lat)) * np.sin(dlon/2)**2)
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

eq["distance_to_bali_km"] = haversine_km(eq["latitude"].values, eq["longitude"].values)

# Flag gempa sangat dangkal (< 5 km): 9 event, perlu inspeksi manual
# tidak di-drop karena bisa legitimate, tapi diflag untuk traceability
eq["depth_shallow_flag"] = (eq["depth_km"] < 5).astype(int)

# Proxy energi seismik: proporsional 10^(1.5 * M) — Gutenberg-Richter
# Magnitude 7.0 melepas energi ~1000x lebih besar dari M5.0,
# sehingga lebih informatif dibanding frekuensi mentah
eq["seismic_energy_proxy"] = 10 ** (1.5 * eq["magnitude"])

# ── Agregasi ke level harian ──────────────────────────────────────────────────
eq_daily = (
    eq.groupby("date")
    .agg(
        eq_count=("event_id", "count"),
        eq_max_magnitude=("magnitude", "max"),
        eq_mean_depth=("depth_km", "mean"),
        eq_seismic_energy=("seismic_energy_proxy", "sum"),
        eq_shallow_flag=("depth_shallow_flag", "max"),  # 1 jika ada event < 5 km di hari itu
    )
)

# ── Resample ke bulanan ───────────────────────────────────────────────────────
eq_monthly = eq_daily.resample("MS").agg(
    eq_count=("eq_count", "sum"),
    eq_max_magnitude=("eq_max_magnitude", "max"),
    eq_mean_depth=("eq_mean_depth", "mean"),
    eq_seismic_energy=("eq_seismic_energy", "sum"),
    eq_shallow_flag=("eq_shallow_flag", "max"),
)

# ── Composite seismic risk score ──────────────────────────────────────────────
# Normalisasi energy proxy (70%) + frekuensi (30%)
# Energy diberi bobot lebih besar karena 1 event M6+ >> 10 event M4
eq_monthly["eq_energy_norm"] = (
    (eq_monthly["eq_seismic_energy"] - eq_monthly["eq_seismic_energy"].min()) /
    (eq_monthly["eq_seismic_energy"].max() - eq_monthly["eq_seismic_energy"].min())
)
eq_monthly["eq_count_norm"] = (
    (eq_monthly["eq_count"] - eq_monthly["eq_count"].min()) /
    (eq_monthly["eq_count"].max() - eq_monthly["eq_count"].min())
)

eq_monthly["eq_risk_score"] = (
    0.7 * eq_monthly["eq_energy_norm"] +
    0.3 * eq_monthly["eq_count_norm"]
)

eq_monthly[["eq_count", "eq_max_magnitude", "eq_seismic_energy", "eq_risk_score"]].head()

,eq_count,eq_max_magnitude,eq_seismic_energy,eq_risk_score
date,,,,
2009-01-01,6,5.1,1.638291e+08,0.013508
2009-02-01,1,4.0,1.000000e+06,0.000000
2009-03-01,1,4.9,2.238721e+07,0.000293
2009-04-01,2,4.8,1.866731e+07,0.002498
2009-05-01,1,4.6,7.943282e+06,0.000095


# Load dan Preprocess Weather

In [ ]:
# Load dan Preprocess Weather (Open-Meteo)

# ── Load file ─────────────────────────────────────────────────────────────────
wx_raw = pd.read_csv("../data/raw/weather/open_meteo_bali_weather_extreme_2009_present.csv")

# ── Parse & bersihkan ────────────────────────────────────────────────────────
wx = wx_raw.copy()
wx["date"] = pd.to_datetime(wx["time"])

# Drop kolom redundant / konstan (verified saat EDA):
# - "rain_sum"   : identik 100% dengan precipitation_sum (no snow di Bali)
# - "latitude"   : konstan -8.400702
# - "longitude"  : konstan 115.184555
# - "timezone"   : konstan Asia/Makassar
# - "source"     : konstan Open-Meteo Historical Weather API
# - "time"       : sudah dipindah ke kolom "date"
wx = wx.drop(columns=["time", "rain_sum", "latitude", "longitude", "timezone", "source"])

# Transformasi distribusi: extreme_weather_score & precipitation_sum
# keduanya sangat zero-inflated (75th percentile = 0 untuk score, banyak hari kering)
# log1p menstabilkan variance tanpa menghilangkan nilai 0
wx["extreme_score_log"] = np.log1p(wx["extreme_weather_score"])
wx["precip_log"]        = np.log1p(wx["precipitation_sum"])

# ── Resample ke bulanan ───────────────────────────────────────────────────────
wx = wx.set_index("date").sort_index()

wx_monthly = (
    wx
    .resample("MS")
    .agg(
        wx_precip_sum=("precipitation_sum", "sum"),
        wx_humidity_mean=("relative_humidity_2m_mean", "mean"),
    )
)

# # ── Composite weather risk score ──────────────────────────────────────────────
# # Cuaca ekstrem → risiko pariwisata tinggi (banjir, pembatalan, wisatawan tidak nyaman)
# # Outlier curah hujan (max 175 mm/hari) dan angin (max 63 km/h) TIDAK di-clip
# # karena konteks tropis Bali — nilai ekstrem ini memang real dan informatif
# def minmax_norm(s):
#     return (s - s.min()) / (s.max() - s.min())

# wx_monthly["wx_precip_norm"]  = minmax_norm(wx_monthly["wx_precip_sum"])
# wx_monthly["wx_extreme_norm"] = minmax_norm(wx_monthly["wx_extreme_days"])
# wx_monthly["wx_wind_norm"]    = minmax_norm(wx_monthly["wx_max_wind_gust"])

# # Composite: 40% curah hujan bulanan, 40% jumlah hari ekstrem, 20% angin
# wx_monthly["wx_risk_score"] = (
#     0.4 * wx_monthly["wx_precip_norm"] +
#     0.4 * wx_monthly["wx_extreme_norm"] +
#     0.2 * wx_monthly["wx_wind_norm"]
# )

# Load dan Preprocess All Weather

In [ ]:
# Load dan Preprocess BalIGuard Historical Monthly Features

# ── Load file ─────────────────────────────────────────────────────────────────
bg_raw = pd.read_csv("../data/raw/weather/baliguard_historical_monthly_features_2009_present.csv")

# ── Parse & set index ─────────────────────────────────────────────────────────
bg = bg_raw.copy()
bg["month"] = pd.to_datetime(bg["month"])
bg = bg.set_index("month").sort_index()

# ── Drop kolom redundant ──────────────────────────────────────────────────────
# weather_max_precipitation identik 100% dengan weather_max_rain (verified via EDA)
bg = bg.drop(columns=["weather_max_precipitation"])

# ── Catatan: pvmbg_volcano_status == max(agung, batur) ───────────────────────
# Merupakan derived column. Disimpan sebagai composite signal tapi JANGAN
# dijumlahkan/digabungkan bebas dengan volcano_status_agung & batur karena
# akan terjadi triple-counting secara implisit.

# ── Seismic energy proxy dari BMKG max magnitude ─────────────────────────────
# 10^(1.5*M): M6.9 melepas ~1000x lebih besar dari M4.9
# Lebih informatif dibanding magnitude mentah untuk risk scoring
bg["bmkg_seismic_energy"] = 10 ** (1.5 * bg["bmkg_max_magnitude_30d"])

# ── Log transform kolom zero-inflated / heavy right-skew ─────────────────────
# earthquake_count: mean=11, max=134 (Lombok 2018 outlier — real, tidak di-clip)
# weather_max_rain: max=175.7 mm — legitimate tropis, tidak di-clip
bg["eq_count_log"]  = np.log1p(bg["earthquake_count"])
bg["eq_m5plus_log"] = np.log1p(bg["earthquake_count_m5_plus"])
bg["rain_log"]      = np.log1p(bg["weather_max_rain"])

# ── Composite disaster risk score ─────────────────────────────────────────────
def minmax_norm(s):
    denom = s.max() - s.min()
    return (s - s.min()) / denom if denom != 0 else pd.Series(0, index=s.index)

# Komponen seismik: energy proxy (magnitude-weighted, bukan frekuensi mentah)
bg["seismic_norm"] = minmax_norm(bg["bmkg_seismic_energy"])

# Komponen gunung berapi: ordinal 0 (normal) → 3 (awas), sudah numerik bermakna
bg["volcano_norm"] = minmax_norm(bg["pvmbg_volcano_status"])

# Komponen cuaca: extreme days (40%) + curah hujan maks (40%) + angin (20%)
bg["weather_severity_norm"] = (
    0.4 * minmax_norm(bg["bmkg_extreme_weather_days"]) +
    0.4 * minmax_norm(bg["weather_max_rain"]) +
    0.2 * minmax_norm(bg["weather_max_wind_gust"])
)

# Composite: 40% seismik + 30% gunung berapi + 30% cuaca
bg["disaster_risk_score"] = (
    0.4 * bg["seismic_norm"] +
    0.3 * bg["volcano_norm"] +
    0.3 * bg["weather_severity_norm"]
)

bg[["bmkg_max_magnitude_30d", "pvmbg_volcano_status",
    "bmkg_extreme_weather_days", "disaster_risk_score"]].head(10)

,bmkg_max_magnitude_30d,pvmbg_volcano_status,bmkg_extreme_weather_days,disaster_risk_score
month,,,,
2009-01-01,5.1,0,6,0.089349
2009-02-01,4.0,0,4,0.092702
2009-03-01,4.9,0,2,0.037040
2009-04-01,4.8,0,1,0.026324
2009-05-01,4.6,0,1,0.033700
2009-06-01,5.5,0,0,0.021086
2009-07-01,5.9,0,0,0.035900
2009-08-01,4.5,0,0,0.030300
2009-09-01,5.7,0,1,0.046395


# Load dan preprocess GDELT

In [ ]:
# ── Load kedua file ───────────────────────────────────────────────────────────
gdelt_hist    = pd.read_csv("../data/raw/gdelt/gdelt_historical_2009_2021.csv")
gdelt_recent  = pd.read_csv("../data/raw/gdelt/gdelt_2026_W24.csv")

# ── Gabungkan & deduplikasi ───────────────────────────────────────────────────
gdelt = (
    pd.concat([gdelt_hist, gdelt_recent])
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .drop_duplicates(subset=["date"], keep="last")  # recent menang kalau overlap
    .set_index("date")
    .sort_index()
)

# ── Load kolom tambahan dari file gdelt baru ──────────────────────────────────
# gdelt_historical_2009_2025.csv  → kolom tambahan: crime_count
# gdelt_historical_env_2009_2025.csv → kolom tambahan: issue_article_count
# Nilai avg_tone, article_count, dll. pada kedua file ini identik dengan gdelt utama.
# Kita hanya mengambil kolom barunya saja, lalu di-join ke gdelt utama by date.

gdelt_crime = (
    pd.read_csv("../data/raw/gdelt/gdelt_historical_2009_2025.csv")
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .set_index("date")
    [["crime_count"]]  # ambil kolom baru saja
)

gdelt_env = (
    pd.read_csv("../data/raw/gdelt/gdelt_historical_env_2009_2025.csv")
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .set_index("date")
    [["issue_article_count"]]  # ambil kolom baru saja
)

# Join ke gdelt utama — left join agar baris gdelt_recent tetap terjaga
gdelt = gdelt.join(gdelt_crime, how="left")
gdelt = gdelt.join(gdelt_env,   how="left")

# Derived: crime ratio & issue ratio terhadap total artikel
gdelt["crime_ratio"]  = gdelt["crime_count"]  / gdelt["article_count"].replace(0, np.nan)
gdelt["issue_ratio"]  = gdelt["issue_article_count"] / gdelt["article_count"].replace(0, np.nan)

# NaN untuk baris di luar coverage file baru (misal: gdelt_recent / 2026)
# → dibiarkan NaN, akan ditangani saat interpolasi di Merge Dataset

# Fitur turunan: rasio artikel risiko
gdelt["risk_ratio"] = gdelt["risk_article_count"] / gdelt["article_count"].replace(0, np.nan)

# Normalisasi tone: makin negatif → makin tinggi risiko → dibalik
gdelt["tone_risk_score"] = (gdelt["avg_tone"] - gdelt["avg_tone"].max()) / \
                           (gdelt["avg_tone"].min() - gdelt["avg_tone"].max())

# Normalisasi risk_ratio
gdelt["risk_ratio_score"] = (gdelt["risk_ratio"] - gdelt["risk_ratio"].min()) / \
                             (gdelt["risk_ratio"].max() - gdelt["risk_ratio"].min())

# Composite GDELT score
gdelt["gdelt_crisis_score"] = (gdelt["tone_risk_score"] + gdelt["risk_ratio_score"]) / 2

gdelt[["avg_tone", "risk_ratio", "tone_risk_score", "risk_ratio_score", "gdelt_crisis_score"]].head()

,avg_tone,risk_ratio,tone_risk_score,risk_ratio_score,gdelt_crisis_score
date,,,,,
2009-01-01,4.881555,0.003690,0.120732,0.030061,0.075397
2009-02-01,4.760496,0.007712,0.131566,0.062827,0.097197
2009-03-01,5.302634,0.068852,0.083048,0.560913,0.321980
2009-04-01,4.192748,0.025490,0.182377,0.207658,0.195018
2009-05-01,5.349026,0.000000,0.078896,0.000000,0.039448


# Load dan Preprocess GTRENDS

In [ ]:
# ── Load & gabungkan ──────────────────────────────────────────────────────────
gtrends_hist     = pd.read_csv("../data/raw/gtrends/gtrends_historical_2009_2021.csv")
gtrends_recent   = pd.read_csv("../data/raw/gtrends/gtrends_2026_W24.csv")
gtrends_combined = pd.read_csv("../data/raw/gtrends/gtrends_positive_combined_historical_2009_2025_2.csv")

gtrends = (
    pd.concat([gtrends_hist, gtrends_recent, gtrends_combined])
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .drop_duplicates(subset=["date"], keep="last")
    .set_index("date")
    .sort_index()
)

gtrends_monthly = gtrends.resample("MS").mean()

# ── Identifikasi keyword columns secara dinamis ───────────────────────────────
# Exclude kolom non-keyword kalau ada (sesuaikan jika ada kolom metadata lain)
NON_KEYWORD_COLS = []  # tambahkan nama kolom metadata di sini kalau ada
keyword_cols = [c for c in gtrends_monthly.columns if c not in NON_KEYWORD_COLS]

print(f"Total keyword columns: {len(keyword_cols)}")
print(keyword_cols)

# ── Coverage check per baris ──────────────────────────────────────────────────
# Penting: tahu berapa keyword yang tersedia per bulan
gtrends_monthly["kw_coverage"] = gtrends_monthly[keyword_cols].notna().sum(axis=1)
gtrends_monthly["kw_coverage_pct"] = gtrends_monthly["kw_coverage"] / len(keyword_cols)

# Flag baris dengan coverage rendah (< 50% keyword tersedia)
# Baris ini composite-nya kurang reliable
gtrends_monthly["low_coverage_flag"] = (gtrends_monthly["kw_coverage_pct"] < 0.5).astype(int)

print("\nCoverage per periode:")
print(gtrends_monthly[["kw_coverage", "kw_coverage_pct", "low_coverage_flag"]].describe().round(2))

# ── Composite: mean dari keyword yang tersedia ────────────────────────────────
# skipna=True by default → otomatis exclude NaN, tapi coverage sudah di-flag di atas
gtrends_monthly["trend_composite"] = gtrends_monthly[keyword_cols].mean(axis=1)

# ── Normalisasi min-max → invert jadi risk score ──────────────────────────────
_min = gtrends_monthly["trend_composite"].min()
_max = gtrends_monthly["trend_composite"].max()

gtrends_monthly["trend_risk_score"] = 1 - (
    (gtrends_monthly["trend_composite"] - _min) / (_max - _min)
)

# ── Drop flag: >25% di bawah rolling mean 3 bulan ────────────────────────────
gtrends_monthly["trend_rolling_mean"] = (
    gtrends_monthly["trend_composite"].rolling(3, min_periods=2).mean()
)
gtrends_monthly["trend_drop_flag"] = (
    gtrends_monthly["trend_composite"] < gtrends_monthly["trend_rolling_mean"] * 0.75
).astype(int)

# ── Output ────────────────────────────────────────────────────────────────────
output_cols = [
    "trend_composite", "trend_risk_score",
    "trend_rolling_mean", "trend_drop_flag",
    "kw_coverage", "kw_coverage_pct", "low_coverage_flag"
]
gtrends_monthly[output_cols].head(10)

Total keyword columns: 26
['Bali vacation', 'Bali travel', 'Bali hotel', 'Ubud Bali', 'Tanah Lot Bali', 'Seminyak Bali', 'Nusa Penida Bali', 'Bali surfing', 'Bali white water rafting', 'Bali food', 'best restaurants Bali', 'Bali cafe', 'Bali cooking class', 'Bali villa private pool', 'Bali resort', 'Bali boutique hotel', 'Bali airbnb', 'Bali yoga retreat', 'Bali spa massage', 'Bali meditation', 'Bali honeymoon', 'Bali waterpark', 'Bali shopping', 'Bali souvenir', 'Bali market', 'Bali fashion']

Coverage per periode:
       kw_coverage  kw_coverage_pct  low_coverage_flag
count       211.00           211.00             211.00
mean         22.43             0.86               0.03
std           3.33             0.13               0.17
min           3.00             0.12               0.00
25%          23.00             0.88               0.00
50%          23.00             0.88               0.00
75%          23.00             0.88               0.00
max          23.00             0.88   

,trend_composite,trend_risk_score,trend_rolling_mean,trend_drop_flag,kw_coverage,kw_coverage_pct,low_coverage_flag
date,,,,,,,
2008-12-01,14.652174,0.817555,NaN,0,23,0.884615,0
2009-01-01,15.739130,0.786474,15.195652,0,23,0.884615,0
2009-02-01,17.413043,0.738609,15.934783,0,23,0.884615,0
2009-03-01,18.686957,0.702182,17.279710,0,23,0.884615,0
2009-04-01,18.717391,0.701312,18.272464,0,23,0.884615,0
2009-05-01,20.443478,0.651955,19.282609,0,23,0.884615,0
2009-06-01,21.978261,0.608069,20.379710,0,23,0.884615,0
2009-07-01,22.608696,0.590042,21.676812,0,23,0.884615,0
2009-08-01,21.017391,0.635544,21.868116,0,23,0.884615,0


# Load dan Preprocess World Bank

In [ ]:
wb = pd.read_csv("../data/raw/worldbank/worldbank_2026-06-10.csv")
wb["date"] = pd.to_datetime(wb["date"])

# Handle 2025: composite_gdp_weighted = 0 → NaN
wb.loc[wb["composite_gdp_weighted"] == 0, "composite_gdp_weighted"] = np.nan

# Pivot per negara
wb_pivot = wb.pivot_table(index="date", columns="country", values="composite_gdp_weighted")

# Interpolasi linear untuk 2025, fallback ffill
wb_pivot = wb_pivot.interpolate(method="linear", limit_direction="forward")
wb_pivot = wb_pivot.ffill()

wb_pivot.head()

country,AUS,CHN,DEU,GBR,IND,JPN,MYS,SGP,USA
date,,,,,,,,,
2009-01-01,1.469468,1.469468,1.469468,1.469468,1.469468,1.469468,1.469468,1.469468,1.469468
2010-01-01,5.627735,5.627735,5.627735,5.627735,5.627735,5.627735,5.627735,5.627735,5.627735
2011-01-01,4.072502,4.072502,4.072502,4.072502,4.072502,4.072502,4.072502,4.072502,4.072502
2012-01-01,4.084882,4.084882,4.084882,4.084882,4.084882,4.084882,4.084882,4.084882,4.084882
2013-01-01,3.861158,3.861158,3.861158,3.861158,3.861158,3.861158,3.861158,3.861158,3.861158


# World Bank: Weighted Economic Index

In [8]:
TOURIST_WEIGHT = {
    "AUS": 0.20,
    "CHN": 0.18,
    "GBR": 0.08,
    "USA": 0.08,
    "DEU": 0.07,
    "JPN": 0.10,
    "IND": 0.12,
    "MYS": 0.10,
    "SGP": 0.07,
}

available_countries = [c for c in TOURIST_WEIGHT if c in wb_pivot.columns]
weights = np.array([TOURIST_WEIGHT[c] for c in available_countries])
weights = weights / weights.sum()

wb_pivot["economic_index"] = wb_pivot[available_countries].values @ weights

# Normalisasi: GDP rendah = risiko tinggi → dibalik
wb_pivot["economic_risk_score"] = 1 - (
    (wb_pivot["economic_index"] - wb_pivot["economic_index"].min()) /
    (wb_pivot["economic_index"].max() - wb_pivot["economic_index"].min())
)

# Expand tahunan → bulanan
wb_monthly = wb_pivot[["economic_index", "economic_risk_score"]].resample("MS").ffill()

wb_monthly.head(14)

country,economic_index,economic_risk_score
date,,
2009-01-01,1.469468,0.520559
2009-02-01,1.469468,0.520559
2009-03-01,1.469468,0.520559
2009-04-01,1.469468,0.520559
2009-05-01,1.469468,0.520559
2009-06-01,1.469468,0.520559
2009-07-01,1.469468,0.520559
2009-08-01,1.469468,0.520559
2009-09-01,1.469468,0.520559


# Merge Dataset

In [9]:
# Merge Dataset

start = max(
    gdelt.index.min(),
    gtrends_monthly.index.min(),
    wb_monthly.index.min(),
    eq_monthly.index.min(),
    wx_monthly.index.min(),
    bg.index.min(),
)
end = min(
    gdelt.index.max(),
    gtrends_monthly.index.max(),
    wb_monthly.index.max(),
    eq_monthly.index.max(),
    wx_monthly.index.max(),
    bg.index.max(),
)

gdelt_sel  = gdelt[["gdelt_crisis_score", "avg_tone", "risk_ratio"]].loc[start:end]
trends_sel = gtrends_monthly[["trend_composite", "trend_risk_score", "trend_drop_flag"]].loc[start:end]
wb_sel     = wb_monthly[["economic_index", "economic_risk_score"]].loc[start:end]
eq_sel     = eq_monthly[["eq_count", "eq_max_magnitude", "eq_seismic_energy", "eq_risk_score"]].loc[start:end]
wx_sel     = wx_monthly[["wx_precip_sum", "wx_humidity_mean"]].loc[start:end]
bg_sel     = bg[[
    # Raw features
    "bmkg_max_magnitude_30d", "bmkg_seismic_energy",
    "earthquake_count", "earthquake_count_m5_plus",
    "pvmbg_volcano_status", "volcano_status_agung", "volcano_status_batur",
    "bmkg_extreme_weather_days", "weather_max_rain",
    "weather_max_wind_speed", "weather_max_wind_gust", "weather_max_temperature",
    # Transformed
    "eq_count_log", "eq_m5plus_log", "rain_log",
    # Composite score
    "disaster_risk_score",
]].loc[start:end]

combined = (
    gdelt_sel
    .join(trends_sel, how="outer")
    .join(wb_sel,     how="outer")
    .join(eq_sel,     how="outer")
    .join(wx_sel,     how="outer")
    .join(bg_sel,     how="outer")
)
combined = combined.sort_index()
combined = combined.interpolate(method="time", limit=2)

print(f"Shape: {combined.shape}")
print(f"Range: {combined.index.min()} → {combined.index.max()}")
print()
print("Nulls:")
print(combined.isnull().sum())

Shape: (181, 30)
Range: 2009-01-01 00:00:00 → 2024-01-01 00:00:00

Nulls:
gdelt_crisis_score           0
avg_tone                     0
risk_ratio                   0
trend_composite              0
trend_risk_score             0
trend_drop_flag              0
economic_index               0
economic_risk_score          0
eq_count                     0
eq_max_magnitude             0
eq_seismic_energy            0
eq_risk_score                0
wx_precip_sum                0
wx_humidity_mean             0
bmkg_max_magnitude_30d       0
bmkg_seismic_energy          0
earthquake_count             0
earthquake_count_m5_plus     0
pvmbg_volcano_status         0
volcano_status_agung         0
volcano_status_batur         0
bmkg_extreme_weather_days    0
weather_max_rain             0
weather_max_wind_speed       0
weather_max_wind_gust        0
weather_max_temperature      0
eq_count_log                 0
eq_m5plus_log                0
rain_log                     0
disaster_risk_score        

# Save

In [ ]:
# Jalankan cell ini hanya kalau sudah siap export
combined.to_csv("../data/processed/combined_additional_features_new.csv.csv")
print("Saved.")

Saved.
